In [1]:
import autogen
autogen.__version__

C:\Users\manju\anaconda3\lib\site-packages\flaml\__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")
C:\Users\manju\anaconda3\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Users\manju\anaconda3\lib\site-packages\autogen\oai\gemini.py:44: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README 

'0.2.35'

In [2]:
#Dowmload ollama
#ollama pull llama3 -- Downloading purpose
#ollama run llama3 -- Prompting
#!pip install pyautogen[ollama] (0.4.7)

In [3]:
import agentops
agentops.init("yourkey")

🖇 AgentOps: [ConcurrentFuturesInstrumentor] Successfully instrumented concurrent.futures.ThreadPoolExecutor
🖇 AgentOps: Unexpected error during span export: 'AuthenticatedOTLPExporter' object has no attribute '_headers'
🖇 AgentOps: Unexpected error during span export: 'AuthenticatedOTLPExporter' object has no attribute '_headers'
🖇 AgentOps: Unexpected error during span export: 'AuthenticatedOTLPExporter' object has no attribute '_headers'
🖇 AgentOps: Unexpected error during span export: 'AuthenticatedOTLPExporter' object has no attribute '_headers'


In [4]:
from pathlib import Path

from autogen import ConversableAgent
from autogen.coding import  CodeBlock, LocalCommandLineCodeExecutor
work_dir = Path("coding")
work_dir.mkdir(exist_ok=True)

executor = LocalCommandLineCodeExecutor(work_dir=work_dir)

#Agent 1
code_executor_agent = ConversableAgent(
    name="code_executor_agent",
    llm_config=False,
    code_execution_config={
        "executor": executor,
    },
    human_input_mode="NEVER",
)

In [5]:
config_list = [
    {
        # Let's choose the Meta's Llama 3.1 model (model names must match Ollama exactly)
        "model": "llama3.1",
        # We specify the API Type as 'ollama' so it uses the Ollama client class
        "api_type": "ollama",
        "stream": False,
        #"client_host": "http://192.168.0.1:11434",
    }
]

In [10]:
import requests

url = "http://127.0.0.1:11434/api/tags"

try:
    response = requests.get(url, timeout=5)
    print("Status:", response.status_code)
    print(response.json())
except Exception as e:
    print("Connection failed:", e)

Status: 200
{'models': [{'name': 'llama3.1:latest', 'model': 'llama3.1:latest', 'modified_at': '2026-08-18T21:33:07.363974+02:00', 'size': 4920753328, 'digest': '46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '8.0B', 'quantization_level': 'Q4_K_M', 'context_length': 131072, 'embedding_length': 4096}, 'capabilities': ['completion', 'tools']}, {'name': 'llama3.1:8b', 'model': 'llama3.1:8b', 'modified_at': '2026-07-29T22:19:33.8441688+02:00', 'size': 4920753328, 'digest': '46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '8.0B', 'quantization_level': 'Q4_K_M', 'context_length': 131072, 'embedding_length': 4096}, 'capabilities': ['completion', 'tools']}]}


In [12]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama"
)

response = client.chat.completions.create(
    model="llama3.1:latest",
    messages=[
        {"role": "user", "content": "Hello! Reply in one sentence."}
    ],
    timeout=120
)

print(response.choices[0].message.content)

I'm ready to help with your question or task.


In [13]:
from autogen import ConversableAgent

config_list = [
    {
        "model": "llama3.1:latest",
        "base_url": "http://127.0.0.1:11434/v1",
        "api_key": "ollama"
    }
]

code_writer_agent = ConversableAgent(
    name="code_writer",
    system_message=code_writer_system_message,
    llm_config={"config_list": config_list},
    code_execution_config=False,
    max_consecutive_auto_reply=2,
    human_input_mode="NEVER"
)

print("Agent created successfully!")

Agent created successfully!


In [14]:
# The code writer agent's system message is to instruct the LLM on how to
# use the Jupyter code executor with IPython kernel.
code_writer_system_message = """
You have been given coding capability to solve tasks using Python code.
In the following cases, suggest python code (in a python coding block) or shell script (in a sh coding block) for the user to execute.
    1. When you need to collect info, use the code to output the info you need, for example, browse or search the web, download/read a file, 
    print the content of a webpage or a file, get the current date/time, check the operating system. After sufficient info is printed and the 
    task is ready to be solved based on your language skill, you can solve the task by yourself.
    2. When you need to perform some task with code, use the code to perform the task and output the result. Finish the task smartly.
Solve the task step by step if you need to. If a plan is not provided, explain your plan first. Be clear which step uses code, and which step uses your 
language skill.When using code, you must indicate the script type in the code block. The user cannot provide any other feedback or perform any other 
action beyond executing the code you suggest. The user can't modify your code. So do not suggest incomplete code which requires users to modify. 
Don't use a code block if it's not intended to be executed by the user.
If you want the user to save the code in a file before executing it, put # filename: <filename> inside the code block as the first line. Don't include multiple code blocks in one response. Do not ask users to copy and paste the result. Instead, use 'print' function for the output when relevant. Check the execution result returned by the user.
"""

import os

#Agent 2

code_writer_agent = ConversableAgent(
    "code_writer",
    system_message=code_writer_system_message,
    llm_config={"config_list":config_list},
    code_execution_config=False,  # Turn off code execution for this agent.
    max_consecutive_auto_reply=2,
    human_input_mode="NEVER",
)

In [16]:
import pprint

chat_result = code_executor_agent.initiate_chat(
    code_writer_agent, message="""Write optimized Python code to calculate the number of unique permutations of the word 'ALGEBRA.'
    For example, the word 'ALGEbRA' should have 2520 unique permutations. Use efficient logic to compute the result and verify the output using 
    the word 'ALGEBRA.' The code should only return the final permutation count."""
)

pprint.pprint(chat_result)
#agentops.end_session("Success")

code_executor_agent (to code_writer):

Write optimized Python code to calculate the number of unique permutations of the word 'ALGEBRA.'
    For example, the word 'ALGEbRA' should have 2520 unique permutations. Use efficient logic to compute the result and verify the output using 
    the word 'ALGEBRA.' The code should only return the final permutation count.

--------------------------------------------------------------------------------
code_writer (to code_executor_agent):

```python
import math

# the total number of permutations, is represented by the factorial of the number of letters in the word
def total_permutations():
    word = "ALGEBRA"
    factorial = math.factorial(len(word))
    return factorial

# however, some letters are repeated (the letter E appears 2 times and the letter A appears 2 times)
def unique_permutations_corrected():
    word = "ALGEBRA"
    factorial_of_unique_elements = math.factorial(len(set(word)))
    denominator = 1
    for letter in set(word):
   

In [17]:
dir(chat_result)


['__annotations__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'chat_history',
 'chat_id',
 'cost',
 'human_input',
 'summary']

In [18]:
chat_result.chat_history

[{'content': "Write optimized Python code to calculate the number of unique permutations of the word 'ALGEBRA.'\n    For example, the word 'ALGEbRA' should have 2520 unique permutations. Use efficient logic to compute the result and verify the output using \n    the word 'ALGEBRA.' The code should only return the final permutation count.",
  'role': 'assistant',
  'name': 'code_executor_agent'},
 {'content': '```python\nimport math\n\n# the total number of permutations, is represented by the factorial of the number of letters in the word\ndef total_permutations():\n    word = "ALGEBRA"\n    factorial = math.factorial(len(word))\n    return factorial\n\n# however, some letters are repeated (the letter E appears 2 times and the letter A appears 2 times)\ndef unique_permutations_corrected():\n    word = "ALGEBRA"\n    factorial_of_unique_elements = math.factorial(len(set(word)))\n    denominator = 1\n    for letter in set(word):\n        denominator *= math.factorial(word.count(letter))\n

In [19]:
chat_result.chat_id

In [20]:
chat_result.cost

{'usage_including_cached_inference': {'total_cost': 0,
  'llama3.1:latest': {'cost': 0,
   'prompt_tokens': 2216,
   'completion_tokens': 1528,
   'total_tokens': 3744}},
 'usage_excluding_cached_inference': {'total_cost': 0,
  'llama3.1:latest': {'cost': 0,
   'prompt_tokens': 1108,
   'completion_tokens': 764,
   'total_tokens': 1872}}}

In [21]:
chat_result.human_input

[]

In [22]:
chat_result.summary

'exitcode: 0 (execution succeeded)\nCode output: 180\n180\n'